In [ ]:
using Random
using Statistics
using Printf
using LinearAlgebra
using Plots
using Logging

function find_project_root(start::AbstractString=pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "System1D.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate project root from $start")
        dir = parent
    end
end

PROJECT_ROOT = find_project_root()

if !isdefined(Main, :nb_paths)
    include(joinpath(PROJECT_ROOT, "Experiments", "common", "notebook_helpers.jl"))
end

NOTEBOOK_REL_DIR = joinpath("Experiments", "systems", "fermion_ring_1d", "dmc", "notebooks")
PATHS = nb_paths(PROJECT_ROOT, NOTEBOOK_REL_DIR)
nb_include_formatting(PATHS.notebook_dir)

if !isdefined(Main, :System1D)
    include(joinpath(PROJECT_ROOT, "src", "System1D.jl"))
end
using .System1D

default(; dpi=170)
nothing


## Model and DMC Parameters

This notebook runs fixed-node DMC for the built-in `SpinlessFermionRing1D` model.

The model represents `N` spinless fermions on a periodic ring with length `L = M * a` and lattice potential
`V(x) = V0 * cos(2*pi*x/a)` applied to each particle coordinate.

Parameters used below:
- Fermion count `N = 2`
- Number of lattice periods `M = 2`
- Lattice spacing `a = 1.0`
- Ring length `L = 2.0`
- Lattice amplitude `V0 = 1.0`
- Twist angle `twist = 0.0`
- Time step `dt = 3.0e-3`
- Total steps `nsteps = 1200`
- Equilibration steps `nequil = 200`
- Target population `targetN = 400`

Trial / node structure:
- Trial state comes from `trial_wavefunction(model)`
- Guiding policy comes from `importance_guiding(model)`
- Node policy is `FixedNode()`


## Julia Construction

The next cell constructs the ring model, derives its Hamiltonian and trial data from the source API, and sets the notebook toggles.

This is the cell to edit if you want a different fermion count, twist, debug cadence, or CSV output name.


In [ ]:
N = 2
M = 2
a = 1.0
L = M * a
V0 = 5.0
twist = 0.0

model = SpinlessFermionRing1D(N, a, L, V0; twist=twist, D=0.5, node_tol=1.0e-7, trig_eps=1.0e-10)
H = hamiltonian(model)
trial = trial_wavefunction(model)
guiding = importance_guiding(model)

targetN = 400
dt = 3.0e-3
nsteps = 1200
nequil = 200
ET0 = -0.2
branch_cap = 2
nblocks = 50

params = DMCParams(; dt=dt, nsteps=nsteps, nequil=nequil, targetN=targetN, ET0=ET0, population_control_gain=2.0, branch_cap=branch_cap, nblocks=nblocks)

rng_init = MersenneTwister(1234)
initial_positions = sample_uniform_configurations(model, targetN, rng_init)

SNAPSHOT_STEPS = nb_default_snapshot_steps(nsteps)
NBINS = 140
DENSITY_SMOOTHING = 9
PERIOD_MARKERS = collect(0.0:a:L)

RUN_LABEL = "guided fixed node"
RUN_COLOR = :navy
PLOT_TITLE = "Spinless fermion ring DMC"
DENSITY_TITLE = "Spinless fermion ring DMC: pooled one-body densities"

SHOW_PROGRESS = true
PROGRESS_EVERY = 0
DEBUG_MODE = true
DEBUG_EVERY = 1
WRITE_RUN_CSV = false
CSV_FILENAME = "spinless_fermion_ring_dmc.csv"
SAVE_FIGURES = false
FIGURE_STEM = "spinless_fermion_ring_dmc"


In [ ]:
sim = run_dmc(
    H,
    params,
    initial_positions;
    rng=MersenneTwister(52),
    guiding=guiding,
    nodepolicy=FixedNode(),
    snapshot_steps=SNAPSHOT_STEPS,
    show_progress=SHOW_PROGRESS,
    progress_every=PROGRESS_EVERY,
    progress_label=RUN_LABEL,
    debug=DEBUG_MODE,
    debug_every=DEBUG_EVERY,
)

start_idx = min(params.nequil + 1, length(sim.energy_mean_history))
mean_energy, sem_energy = nb_mean_sem(sim.energy_mean_history[start_idx:end])

println(@sprintf("%s mean energy after nequil=%d: %.8f +/- %.3e", RUN_LABEL, params.nequil, mean_energy, sem_energy))
println("final walker population = ", sim.population_history[end])

if WRITE_RUN_CSV
    csv_path = joinpath(PATHS.tables_dir, CSV_FILENAME)
    nb_write_csv(csv_path, nb_dmc_rows(RUN_LABEL, sim))
    println("Wrote run CSV to: ", abspath(csv_path))
end


In [ ]:
function pooled_ring_coordinates(snapshot)
    xs = Float64[]
    for R in snapshot
        append!(xs, Float64.(R))
    end
    return xs
end

history_fig = nb_plot_dmc_history([sim]; labels=[RUN_LABEL], colors=[RUN_COLOR], title_prefix=PLOT_TITLE)
display(history_fig)
nb_save_figure(history_fig, PATHS.figures_dir, FIGURE_STEM, "history"; enabled=SAVE_FIGURES)

available_steps = SNAPSHOT_STEPS[1:min(length(SNAPSHOT_STEPS), length(sim.walker_positions_history))]
density_fig = plot(
    xlabel="x",
    ylabel="pooled one-body density",
    title=DENSITY_TITLE,
    legend=:topright,
    xlims=(0.0, L),
)
for (snapshot, step_idx) in zip(sim.walker_positions_history, available_steps)
    xs = pooled_ring_coordinates(snapshot)
    centers, density = nb_density_curve(xs; nbins=NBINS, xmin=0.0, xmax=L, smoothing_window=DENSITY_SMOOTHING)
    plot!(density_fig, centers, density; label="step $(step_idx)", color=RUN_COLOR, linewidth=2.2, alpha=0.82)
end
for (k, xmark) in enumerate(PERIOD_MARKERS)
    vline!(density_fig, [xmark]; color=:gray80, linestyle=:dot, linewidth=1.0, label=(k == 1 ? "cell markers" : ""))
end

display(density_fig)
nb_save_figure(density_fig, PATHS.figures_dir, FIGURE_STEM, "density"; enabled=SAVE_FIGURES)
